# عملگرهای گرادیان / Gradient Operators

**دانشجو / Student:** مسیح معافی / Masih Moafi  
**تاریخ / Date:** ۱۴۰۳/۰۷/۲۸ (October 19, 2025)

---

## هدف / Objective

**هدف:** یادگیری عملگرهای گرادیان برای تشخیص لبه‌ها در تصاویر

**Objective:** Learn gradient operators for edge detection in images

---

## محتوا / Contents

1. مقدمه‌ای بر گرادیان تصویر / Introduction to Image Gradients
2. عملگر سوبل / Sobel Operator
3. عملگر شار / Scharr Operator
4. عملگر لاپلاسین / Laplacian Operator
5. لاپلاسین گاوسی (LoG) / Laplacian of Gaussian
6. مقایسه عملگرها / Comparison of Operators
7. تحلیل قدرت و جهت لبه / Edge Strength and Direction Analysis

In [ ]:
# وارد کردن کتابخانه‌های مورد نیاز / Import required libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List
import time

# تنظیمات نمایش / Display settings
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 12

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. توابع کمکی / Helper Functions

In [ ]:
def display_images(images: list, titles: list, cmap='gray', rows=1):
    """
    نمایش چند تصویر در کنار هم
    
    Args:
        images: لیست تصاویر
        titles: لیست عناوین
        cmap: نقشه رنگی
        rows: تعداد ردیف‌ها
    """
    n = len(images)
    cols = (n + rows - 1) // rows
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    
    if n == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if rows > 1 or cols > 1 else [axes]
    
    for i, (img, title) in enumerate(zip(images, titles)):
        axes[i].imshow(img, cmap=cmap)
        axes[i].set_title(title, fontsize=14, fontweight='bold')
        axes[i].axis('off')
    
    # حذف محورهای اضافی
    for i in range(n, len(axes)):
        axes[i].remove()
    
    plt.tight_layout()
    plt.show()


def normalize_image(image: np.ndarray) -> np.ndarray:
    """
    نرمال‌سازی تصویر به بازه [0, 255]
    
    Args:
        image: تصویر ورودی
    
    Returns:
        تصویر نرمال‌شده
    """
    normalized = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX)
    return normalized.astype(np.uint8)


def calculate_gradient_magnitude(gx: np.ndarray, gy: np.ndarray) -> np.ndarray:
    """
    محاسبه قدرت گرادیان
    
    Args:
        gx: گرادیان در جهت x
        gy: گرادیان در جهت y
    
    Returns:
        قدرت گرادیان
    """
    magnitude = np.sqrt(gx.astype(np.float64)**2 + gy.astype(np.float64)**2)
    return magnitude


def calculate_gradient_direction(gx: np.ndarray, gy: np.ndarray) -> np.ndarray:
    """
    محاسبه جهت گرادیان
    
    Args:
        gx: گرادیان در جهت x
        gy: گرادیان در جهت y
    
    Returns:
        جهت گرادیان (به رادیان)
    """
    direction = np.arctan2(gy, gx)
    return direction

print("✓ توابع کمکی آماده شدند")

## 2. ایجاد تصویر نمونه / Create Sample Image

In [ ]:
# ایجاد تصویر نمونه
# Create sample image
image = np.zeros((400, 600), dtype=np.uint8)

# اضافه کردن مستطیل‌ها
# Add rectangles
cv2.rectangle(image, (50, 50), (200, 200), 255, -1)
cv2.rectangle(image, (250, 100), (400, 250), 200, -1)
cv2.rectangle(image, (450, 150), (550, 300), 150, -1)

# اضافه کردن دایره‌ها
# Add circles
cv2.circle(image, (150, 300), 60, 180, -1)
cv2.circle(image, (400, 320), 50, 220, -1)

# اعمال محو گاوسی برای کاهش نویز
# Apply Gaussian blur to reduce noise
image_blurred = cv2.GaussianBlur(image, (5, 5), 0)

display_images([image, image_blurred], 
               ['تصویر اصلی / Original', 'تصویر محو شده / Blurred'])

print(f"اندازه تصویر / Image size: {image.shape}")

## 3. عملگر سوبل / Sobel Operator

عملگر سوبل از مشتق مرتبه اول برای تشخیص لبه استفاده می‌کند.

The Sobel operator uses first-order derivatives for edge detection.

In [ ]:
# محاسبه گرادیان سوبل در جهت x و y
# Calculate Sobel gradients in x and y directions
sobel_x = cv2.Sobel(image_blurred, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(image_blurred, cv2.CV_64F, 0, 1, ksize=3)

# محاسبه قدرت گرادیان
# Calculate gradient magnitude
sobel_magnitude = calculate_gradient_magnitude(sobel_x, sobel_y)

# محاسبه جهت گرادیان
# Calculate gradient direction
sobel_direction = calculate_gradient_direction(sobel_x, sobel_y)

# نرمال‌سازی برای نمایش
# Normalize for display
sobel_x_display = normalize_image(np.abs(sobel_x))
sobel_y_display = normalize_image(np.abs(sobel_y))
sobel_magnitude_display = normalize_image(sobel_magnitude)

# نمایش نتایج
# Display results
images = [image_blurred, sobel_x_display, sobel_y_display, sobel_magnitude_display]
titles = ['تصویر اصلی\nOriginal', 
          'گرادیان X\nGradient X',
          'گرادیان Y\nGradient Y',
          'قدرت گرادیان\nMagnitude']

display_images(images, titles, rows=2)

print("✓ عملگر سوبل اعمال شد")
print(f"محدوده گرادیان X / X gradient range: [{sobel_x.min():.2f}, {sobel_x.max():.2f}]")
print(f"محدوده گرادیان Y / Y gradient range: [{sobel_y.min():.2f}, {sobel_y.max():.2f}]")

### تأثیر اندازه کرنل / Effect of Kernel Size

In [ ]:
# مقایسه اندازه‌های مختلف کرنل
# Compare different kernel sizes
kernel_sizes = [3, 5, 7]
results = []
titles = ['تصویر اصلی\nOriginal']

for ksize in kernel_sizes:
    sobel_x = cv2.Sobel(image_blurred, cv2.CV_64F, 1, 0, ksize=ksize)
    sobel_y = cv2.Sobel(image_blurred, cv2.CV_64F, 0, 1, ksize=ksize)
    magnitude = calculate_gradient_magnitude(sobel_x, sobel_y)
    results.append(normalize_image(magnitude))
    titles.append(f'کرنل {ksize}x{ksize}\nKernel {ksize}x{ksize}')

all_images = [image_blurred] + results
display_images(all_images, titles, rows=2)

print("✓ تأثیر اندازه کرنل نمایش داده شد")

## 4. عملگر شار / Scharr Operator

عملگر شار دقت بهتری نسبت به سوبل برای کرنل 3x3 دارد.

The Scharr operator has better accuracy than Sobel for 3x3 kernels.

In [ ]:
# محاسبه گرادیان شار
# Calculate Scharr gradients
scharr_x = cv2.Scharr(image_blurred, cv2.CV_64F, 1, 0)
scharr_y = cv2.Scharr(image_blurred, cv2.CV_64F, 0, 1)

# محاسبه قدرت گرادیان
# Calculate gradient magnitude
scharr_magnitude = calculate_gradient_magnitude(scharr_x, scharr_y)

# نرمال‌سازی برای نمایش
# Normalize for display
scharr_x_display = normalize_image(np.abs(scharr_x))
scharr_y_display = normalize_image(np.abs(scharr_y))
scharr_magnitude_display = normalize_image(scharr_magnitude)

# نمایش نتایج
# Display results
images = [image_blurred, scharr_x_display, scharr_y_display, scharr_magnitude_display]
titles = ['تصویر اصلی\nOriginal',
          'شار X\nScharr X',
          'شار Y\nScharr Y',
          'قدرت گرادیان\nMagnitude']

display_images(images, titles, rows=2)

print("✓ عملگر شار اعمال شد")

### مقایسه سوبل و شار / Comparison of Sobel and Scharr

In [ ]:
# محاسبه سوبل 3x3
# Calculate Sobel 3x3
sobel_x_3 = cv2.Sobel(image_blurred, cv2.CV_64F, 1, 0, ksize=3)
sobel_y_3 = cv2.Sobel(image_blurred, cv2.CV_64F, 0, 1, ksize=3)
sobel_mag_3 = calculate_gradient_magnitude(sobel_x_3, sobel_y_3)

# مقایسه
# Comparison
images = [normalize_image(sobel_mag_3), normalize_image(scharr_magnitude)]
titles = ['سوبل 3x3\nSobel 3x3', 'شار\nScharr']

display_images(images, titles)

# محاسبه تفاوت
# Calculate difference
diff = np.abs(sobel_mag_3 - scharr_magnitude)
print(f"میانگین تفاوت / Mean difference: {diff.mean():.2f}")
print(f"حداکثر تفاوت / Max difference: {diff.max():.2f}")

## 5. عملگر لاپلاسین / Laplacian Operator

عملگر لاپلاسین از مشتق مرتبه دوم استفاده می‌کند و به نویز حساس است.

The Laplacian operator uses second-order derivatives and is sensitive to noise.

In [ ]:
# محاسبه لاپلاسین
# Calculate Laplacian
laplacian = cv2.Laplacian(image_blurred, cv2.CV_64F, ksize=3)

# نرمال‌سازی برای نمایش
# Normalize for display
laplacian_display = normalize_image(np.abs(laplacian))

# نمایش نتایج
# Display results
images = [image_blurred, laplacian_display]
titles = ['تصویر اصلی\nOriginal', 'لاپلاسین\nLaplacian']

display_images(images, titles)

print("✓ عملگر لاپلاسین اعمال شد")
print(f"محدوده لاپلاسین / Laplacian range: [{laplacian.min():.2f}, {laplacian.max():.2f}]")

### حساسیت به نویز / Sensitivity to Noise

In [ ]:
# اضافه کردن نویز به تصویر
# Add noise to image
noise = np.random.normal(0, 10, image.shape)
noisy_image = np.clip(image.astype(np.float64) + noise, 0, 255).astype(np.uint8)

# لاپلاسین بدون محو
# Laplacian without blur
laplacian_noisy = cv2.Laplacian(noisy_image, cv2.CV_64F, ksize=3)

# لاپلاسین با محو
# Laplacian with blur
noisy_blurred = cv2.GaussianBlur(noisy_image, (5, 5), 0)
laplacian_blurred = cv2.Laplacian(noisy_blurred, cv2.CV_64F, ksize=3)

# نمایش نتایج
# Display results
images = [noisy_image, 
          normalize_image(np.abs(laplacian_noisy)),
          normalize_image(np.abs(laplacian_blurred))]
titles = ['تصویر نویزی\nNoisy Image',
          'لاپلاسین (بدون محو)\nLaplacian (no blur)',
          'لاپلاسین (با محو)\nLaplacian (with blur)']

display_images(images, titles)

print("✓ حساسیت به نویز نمایش داده شد")

## 6. لاپلاسین گاوسی (LoG) / Laplacian of Gaussian

ترکیب محو گاوسی و لاپلاسین برای تشخیص لبه مقاوم به نویز

Combining Gaussian blur and Laplacian for noise-robust edge detection

In [ ]:
def laplacian_of_gaussian(image: np.ndarray, sigma: float = 1.0) -> np.ndarray:
    """
    محاسبه لاپلاسین گاوسی
    
    Args:
        image: تصویر ورودی
        sigma: انحراف معیار گاوسی
    
    Returns:
        نتیجه LoG
    """
    # محاسبه اندازه کرنل
    # Calculate kernel size
    ksize = int(6 * sigma + 1)
    if ksize % 2 == 0:
        ksize += 1
    
    # اعمال محو گاوسی
    # Apply Gaussian blur
    blurred = cv2.GaussianBlur(image, (ksize, ksize), sigma)
    
    # محاسبه لاپلاسین
    # Calculate Laplacian
    log = cv2.Laplacian(blurred, cv2.CV_64F)
    
    return log


# محاسبه LoG با سیگماهای مختلف
# Calculate LoG with different sigmas
sigmas = [0.5, 1.0, 2.0]
results = []
titles = ['تصویر نویزی\nNoisy Image']

for sigma in sigmas:
    log = laplacian_of_gaussian(noisy_image, sigma)
    results.append(normalize_image(np.abs(log)))
    titles.append(f'LoG (σ={sigma})\nLoG (σ={sigma})')

all_images = [noisy_image] + results
display_images(all_images, titles, rows=2)

print("✓ لاپلاسین گاوسی محاسبه شد")

## 7. مقایسه عملگرها / Comparison of Operators

In [ ]:
# محاسبه همه عملگرها
# Calculate all operators
sobel_x = cv2.Sobel(image_blurred, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(image_blurred, cv2.CV_64F, 0, 1, ksize=3)
sobel_result = calculate_gradient_magnitude(sobel_x, sobel_y)

scharr_x = cv2.Scharr(image_blurred, cv2.CV_64F, 1, 0)
scharr_y = cv2.Scharr(image_blurred, cv2.CV_64F, 0, 1)
scharr_result = calculate_gradient_magnitude(scharr_x, scharr_y)

laplacian_result = cv2.Laplacian(image_blurred, cv2.CV_64F, ksize=3)

log_result = laplacian_of_gaussian(image, sigma=1.0)

# نمایش مقایسه
# Display comparison
images = [image,
          normalize_image(sobel_result),
          normalize_image(scharr_result),
          normalize_image(np.abs(laplacian_result)),
          normalize_image(np.abs(log_result))]
titles = ['تصویر اصلی\nOriginal',
          'سوبل\nSobel',
          'شار\nScharr',
          'لاپلاسین\nLaplacian',
          'LoG']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, (img, title) in enumerate(zip(images, titles)):
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(title, fontsize=14, fontweight='bold')
    axes[i].axis('off')

axes[-1].remove()

plt.tight_layout()
plt.show()

print("✓ مقایسه عملگرها انجام شد")

## 8. تحلیل قدرت و جهت لبه / Edge Strength and Direction Analysis

In [ ]:
# محاسبه گرادیان
# Calculate gradients
gx = cv2.Sobel(image_blurred, cv2.CV_64F, 1, 0, ksize=3)
gy = cv2.Sobel(image_blurred, cv2.CV_64F, 0, 1, ksize=3)

# محاسبه قدرت و جهت
# Calculate magnitude and direction
magnitude = calculate_gradient_magnitude(gx, gy)
direction = calculate_gradient_direction(gx, gy)

# تبدیل جهت به درجه
# Convert direction to degrees
direction_degrees = np.degrees(direction)

# نمایش نتایج
# Display results
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# تصویر اصلی
axes[0, 0].imshow(image_blurred, cmap='gray')
axes[0, 0].set_title('تصویر اصلی\nOriginal', fontweight='bold')
axes[0, 0].axis('off')

# قدرت گرادیان
im1 = axes[0, 1].imshow(magnitude, cmap='hot')
axes[0, 1].set_title('قدرت گرادیان\nGradient Magnitude', fontweight='bold')
axes[0, 1].axis('off')
plt.colorbar(im1, ax=axes[0, 1])

# جهت گرادیان
im2 = axes[1, 0].imshow(direction_degrees, cmap='hsv')
axes[1, 0].set_title('جهت گرادیان (درجه)\nGradient Direction (degrees)', fontweight='bold')
axes[1, 0].axis('off')
plt.colorbar(im2, ax=axes[1, 0])

# هیستوگرام جهت
axes[1, 1].hist(direction_degrees.ravel(), bins=36, range=(-180, 180))
axes[1, 1].set_title('هیستوگرام جهت\nDirection Histogram', fontweight='bold')
axes[1, 1].set_xlabel('جهت (درجه) / Direction (degrees)')
axes[1, 1].set_ylabel('تعداد / Count')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ تحلیل قدرت و جهت لبه انجام شد")
print(f"میانگین قدرت / Mean magnitude: {magnitude.mean():.2f}")
print(f"حداکثر قدرت / Max magnitude: {magnitude.max():.2f}")

## 9. تست روی تصویر واقعی / Test on Real Image

In [ ]:
# سعی در خواندن تصویر از فصل‌های قبلی
# Try to read image from previous chapters
import os

test_paths = [
    '../chapter0_opencv_tutorial/New_Zealand_Lake.jpg',
    '../chapter0_opencv_tutorial/coca-cola-logo.png',
    '../chapter2_practical/how-many-horses.webp'
]

for path in test_paths:
    if os.path.exists(path):
        # خواندن تصویر
        # Read image
        test_img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        
        if test_img is not None:
            # تغییر اندازه در صورت نیاز
            # Resize if needed
            if test_img.shape[0] > 500 or test_img.shape[1] > 500:
                scale = 500 / max(test_img.shape)
                test_img = cv2.resize(test_img, None, fx=scale, fy=scale)
            
            # اعمال محو
            # Apply blur
            test_blurred = cv2.GaussianBlur(test_img, (5, 5), 0)
            
            # محاسبه عملگرها
            # Calculate operators
            sobel_x = cv2.Sobel(test_blurred, cv2.CV_64F, 1, 0, ksize=3)
            sobel_y = cv2.Sobel(test_blurred, cv2.CV_64F, 0, 1, ksize=3)
            sobel_mag = calculate_gradient_magnitude(sobel_x, sobel_y)
            
            scharr_x = cv2.Scharr(test_blurred, cv2.CV_64F, 1, 0)
            scharr_y = cv2.Scharr(test_blurred, cv2.CV_64F, 0, 1)
            scharr_mag = calculate_gradient_magnitude(scharr_x, scharr_y)
            
            laplacian = cv2.Laplacian(test_blurred, cv2.CV_64F, ksize=3)
            
            # نمایش نتایج
            # Display results
            images = [test_img,
                      normalize_image(sobel_mag),
                      normalize_image(scharr_mag),
                      normalize_image(np.abs(laplacian))]
            titles = ['تصویر اصلی\nOriginal',
                      'سوبل\nSobel',
                      'شار\nScharr',
                      'لاپلاسین\nLaplacian']
            
            display_images(images, titles, rows=2)
            
            print(f"✓ تست روی تصویر واقعی انجام شد: {os.path.basename(path)}")
            break
else:
    print("⚠ تصویر تست یافت نشد")

## 10. تمرین‌ها / Exercises

### تمرین 1 / Exercise 1
تصویری با لبه‌های افقی و عمودی ایجاد کنید و ببینید کدام عملگر (Sobel X یا Sobel Y) هر نوع لبه را بهتر تشخیص می‌دهد.

Create an image with horizontal and vertical edges and see which operator (Sobel X or Sobel Y) detects each type better.

### تمرین 2 / Exercise 2
اندازه‌های مختلف کرنل سوبل (3، 5، 7) را روی یک تصویر نویزی امتحان کنید و نتایج را مقایسه کنید.

Try different Sobel kernel sizes (3, 5, 7) on a noisy image and compare the results.

### تمرین 3 / Exercise 3
یک تابع بنویسید که بر اساس قدرت گرادیان، لبه‌های قوی را از لبه‌های ضعیف جدا کند.

Write a function that separates strong edges from weak edges based on gradient magnitude.

### تمرین 4 / Exercise 4
جهت گرادیان را برای یک تصویر محاسبه کنید و لبه‌ها را بر اساس جهت رنگ‌آمیزی کنید.

Calculate gradient direction for an image and color the edges based on their direction.